Biometrich Technologies and Behavioural Security
# **<center>Tutorial 7 - Behavioral biometric: EEG signal </center>**
### <center> Part 1 </center>

##Step 1: Download Dataset and install MNE

The "<a href=https://physionet.org/content/eegmmidb/1.0.0/>EEG Motor Movement/Imagery Dataset</a>" consists of over 1500 one- and two-minute EEG recordings, obtained from 109 volunteers.

Subjects performed different motor/imagery tasks while 64-channel EEG were recorded using the <a href = "http://www.bci2000.org">BCI2000</a> system. Each subject performed 14 experimental runs: two one-minute baseline runs (one with eyes open, one with eyes closed), and three two-minute runs of each of the four following tasks:

A target appears on either the left or the right side of the screen.

1.   The subject opens and closes the corresponding fist until the target disappears. Then the subject relaxes.
2.   A target appears on either the left or the right side of the screen. The subject imagines opening and closing the corresponding fist until the target disappears. Then the subject relaxes.
3. A target appears on either the top or the bottom of the screen. The subject opens and closes either both fists (if the target is on top) or both feet (if the target is on the bottom) until the target disappears. Then the subject relaxes.
4. A target appears on either the top or the bottom of the screen. The subject imagines opening and closing either both fists (if the target is on top) or both feet (if the target is on the bottom) until the target disappears. Then the subject relaxes.


The data are provided here in EDF+ format (containing 64 EEG signals, each sampled at 160 samples per second, and an annotation channel).
Each annotation includes one of three codes (T0, T1, or T2):

*   T0 corresponds to rest.
*   T1 corresponds to onset of motion (real or imagined) of
the left fist (in runs 3, 4, 7, 8, 11, and 12)
both fists (in runs 5, 6, 9, 10, 13, and 14).
*   T2 corresponds to onset of motion (real or imagined) of
the right fist (in runs 3, 4, 7, 8, 11, and 12)
both feet (in runs 5, 6, 9, 10, 13, and 14).

<font size = 5>**As we said, we use EEG records from T0 Task.**

In [ ]:
import cv2
import numpy as np
import os
from matplotlib import pyplot as plt
from scipy import signal
import scipy.io
import math
import sys
import time
from IPython.display import HTML, display
from IPython.display import clear_output

#Loading part
from pydrive.auth import GoogleAuth
from pydrive.drive import GoogleDrive
from google.colab import auth
from oauth2client.client import GoogleCredentials
auth.authenticate_user()
gauth = GoogleAuth()
gauth.credentials = GoogleCredentials.get_application_default()
drive = GoogleDrive(gauth)

#Dataset download code
downloaded = drive.CreateFile({'id': "1WwuAh25Jfx-I8rY3vFGyXiI79YfLYUpH"})
downloaded.GetContentFile('EEG_T0.zip')
!unzip EEG_T0
clear_output()
print('Done!')

Done!


Feel free to explore <a href = "https://mne.tools/stable/index.html">MNE webpage </a> for further informations.

In [ ]:
pip install mne

     |████████████████████████████████| 7.5 MB 4.6 MB/s 


## Step 2: Signal filtering and splitting into patches

First of all, we need to extract the correct band from raw signals. To do this, we can exploit the **filter** method of MNE.

Once we have the **broadband EEG**, we can split the dataset in order to perform classification:


*   Firsts **50 seconds** of each user are used to build the train set.
*   The remaining part is chosen as the test set.

Finally, we split the EEG record further into patches. The sample rate is **160 Hz**, so that we can choose as a reasonable value a patch length of **160 samples** (1 second).

Summing up:

*   Train set: **50** patches per user.
*   Test set:  **10** patches per user.



In [ ]:
import mne

train_f = os.listdir('/content/EEG_T0')
train = []
label_tr = []
test = []
label_ts = []

# Patch length --> 160 samples == 1 second

l_patch =80
classe = 0
n_utenti = 20

path = '/content/EEG_T0/'

# Split into 5 frequency bands

high = {'alpha' : 13, 'beta' : 30, 'delta' : 4, 'gamma' : 40, 'theta' : 8, 'broadband' : None } # High frequencies
low = {'alpha' : 8, 'beta' : 13, 'delta' : 0.5, 'gamma' : 30, 'theta' : 4, 'broadband' : 1 } # Low frequencies


ntr = 50 # train set window
nts = 10 # test set window

for i in train_f[:n_utenti]:

  r0 = mne.io.read_raw_edf(input_fname = path+i, preload = True, verbose = 'CRITICAL')
  eegobjects = (mne.io.Raw.filter(r0, l_freq = low['broadband'], h_freq = high['broadband'], n_jobs = 8, verbose = 'CRITICAL'))
  registrazione = eegobjects.get_data()

  h = l_patch
  l = 0

  while h <= (ntr*160):
    train.append(registrazione[:,l:h].reshape(64*l_patch))
    label_tr.append(classe)
    l = l + l_patch
    h = h + l_patch

  while h <= ((ntr+nts)*160):
    test.append(registrazione[:,l:h].reshape(64*l_patch))
    l = l + l_patch
    h = h + l_patch
    label_ts.append(classe)

  classe = classe + 1



## Step 3: Feature Extraction

###Step 3.1 Define LBP function

In [ ]:
def get_pixel(img, center, x, y):
    new_value = 0
    try:
        if img[x][y] >= center:
            new_value = 1
    except:
        pass
    return new_value

def lbp_calculated_pixel(img, x, y):

    center = img[x][y]
    val_ar = []
    val_ar.append(get_pixel(img, center, x-1, y+1))     # top_right
    val_ar.append(get_pixel(img, center, x, y+1))       # right
    val_ar.append(get_pixel(img, center, x+1, y+1))     # bottom_right
    val_ar.append(get_pixel(img, center, x+1, y))       # bottom
    val_ar.append(get_pixel(img, center, x+1, y-1))     # bottom_left
    val_ar.append(get_pixel(img, center, x, y-1))       # left
    val_ar.append(get_pixel(img, center, x-1, y-1))     # top_left
    val_ar.append(get_pixel(img, center, x-1, y))       # top

    power_val = [1, 2, 4, 8, 16, 32, 64, 128]
    val = 0
    for i in range(len(val_ar)):
        val += val_ar[i] * power_val[i]
    return val

###Step 3.2 Compute LBP histogram of Train and Test set

In [ ]:
X_tr = []
for k in train:
  img_lbp = np.zeros((64, l_patch,3), np.uint8)
  for i in range(0, 64):
    for j in range(0, l_patch):
      img_lbp[i, j] = lbp_calculated_pixel(k.reshape(64,l_patch), i, j)
  hist_lbp = cv2.calcHist([img_lbp], [0], None, [256], [0, 256])
  hist_lbp = hist_lbp.astype("float")
  hist_lbp /= (hist_lbp.sum() + 1e-7)
  X_tr.append(hist_lbp)
X_tr = np.array(X_tr)[:,:,0]

In [ ]:
X_ts = []
for k in test:
  img_lbp = np.zeros((64, l_patch,3), np.uint8)
  for i in range(0, 64):
    for j in range(0, l_patch):
      img_lbp[i, j] = lbp_calculated_pixel(k.reshape(64,l_patch), i, j)
  hist_lbp = cv2.calcHist([img_lbp], [0], None, [256], [0, 256])
  hist_lbp = hist_lbp.astype("float")
  hist_lbp /= (hist_lbp.sum() + 1e-7)
  X_ts.append(hist_lbp)
X_ts = np.array(X_ts)[:,:,0]

## Step 4: Model definition

Choose one of the approach that you know (or may be a new one) to classify EEG patches and test the performance. Feel free to vary parameters like the patch length, train and test composition, etc.

### Step 4.1 SVM

In [ ]:
from sklearn import svm
clf = svm.SVC()
clf.fit(X_tr, label_tr)
y_pred = clf.predict(X_ts)
from sklearn.metrics import accuracy_score
acc='SVM: Total Accuracy: %.2f %%' % (accuracy_score(label_ts, y_pred)*100)
print(acc)

SVM: Total Accuracy: 76.00 %


### Step 4.2 KNN

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
neigh = KNeighborsClassifier(n_neighbors=3)
neigh.fit(X_tr, label_tr)
y_pred = neigh.predict(X_ts)
acc= 'KNN: Total Accuracy: %.2f %%' % (accuracy_score(label_ts, y_pred)*100)
print(acc)

KNN: Total Accuracy: 74.75 %


### Step 4.3 Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier

clf = RandomForestClassifier()
clf.fit(X_tr, label_tr)
y_pred = clf.predict(X_ts)
from sklearn.metrics import accuracy_score
acc='RF: Total Accuracy: %.2f %%' % (accuracy_score(label_ts, y_pred)*100)
print(acc)

RF: Total Accuracy: 83.75 %
